In [20]:
import pandas as pd
from pathlib import Path


def err(msg):
    print('Error: ' + msg)
    exit(0)
    
def loadNormData(path):
    rawData = []
    goldData = []
    curSent = []

    for line in open(path):
        tok = line.strip().split('\t')

        if tok == [''] or tok == []:
            rawData.append([x[0] for x in curSent])
            goldData.append([x[1] for x in curSent])
            curSent = []

        else:
            if len(tok) > 2:
                err('erroneous input, line:\n' + line + '\n in file ' + path + ' contains more then two elements')
            if len(tok) == 1:
                tok.append('')
            curSent.append(tok)

    # in case file does not end with newline
    if curSent != []:
        rawData.append([x[0] for x in curSent])
        goldData.append([x[1] for x in curSent])
    return rawData, goldData


datasets = {}
for p in Path("./datasets").rglob("*"):
    if p.is_file():
        name = f"{p.parent.name}-{p.name.replace('.norm', '')}"
        raw, norm = loadNormData(p)
        datasets[name] = [{"raw": r, "norm": n} for r,n in list(zip(raw, norm))]

pd.DataFrame(datasets['en-dev']).head(3)

,raw,norm
0,"[@cdutra5, bruh, get, out, yo, feelings, lol]","[@cdutra5, brother, get, out, your, feelings, ..."
1,"[rt, @demberel_s, :, manan, dund, xaragdax, te...","[rt, @demberel_s, :, manan, dund, xaragdax, te..."
2,"[why, dese, niggas, think, dey, doin, summn]","[why, these, niggers, think, they, doing, some..."


In [21]:
from datasets import Dataset, DatasetDict, concatenate_datasets

splits = {"train": [], "validation": [], "test": []}

for k, ds in datasets.items():
    lang, split = k.split("-", 1)  # e.g. en-train, de-dev
    split = "validation" if split == "dev" else split

    if split in splits:
        ds = Dataset.from_list(ds)
        ds = ds.add_column("lang", [lang] * len(ds))
        splits[split].append(ds)

out = DatasetDict({
    s: concatenate_datasets(dsets)
    for s, dsets in splits.items()
    if dsets
})

In [22]:
datasets.keys()

dict_keys(['da-test', 'da-train', 'de-dev', 'de-test', 'de-train', 'en-dev', 'en-test', 'en-train', 'es-test', 'es-train', 'hr-dev', 'hr-test', 'hr-train', 'id-dev', 'id-test', 'id-train', 'iden-dev', 'iden-test', 'iden-train', 'it-test', 'it-train', 'ja-dev', 'ja-test', 'ja-train', 'nl-dev', 'nl-test', 'nl-train', 'sl-dev', 'sl-test', 'sl-train', 'sr-dev', 'sr-test', 'sr-train', 'th-dev', 'th-test', 'th-train', 'tr-test', 'tr-train', 'trde-test', 'trde-train', 'vi-dev', 'vi-test', 'vi-train', 'ko-dev', 'ko-test', 'ko-train'])

## Upload data

In [23]:
DatasetDict({
    "train": out["train"],
    "validation": out["validation"],
}).push_to_hub("weerayut/multilexnorm2026-pub", private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/40 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Upload 0 LFS files: 0it [00:00, ?it/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Upload 0 LFS files: 0it [00:00, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/weerayut/multilexnorm2026-pub/commit/7124c792404e517869fc44dd2859f7acd3d7523f', commit_message='Upload dataset', commit_description='', oid='7124c792404e517869fc44dd2859f7acd3d7523f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/weerayut/multilexnorm2026-pub', endpoint='https://huggingface.co', repo_type='dataset', repo_id='weerayut/multilexnorm2026-pub'), pr_revision=None, pr_num=None)

In [24]:
DatasetDict({
    "test": out["test"],
}).push_to_hub("weerayut/multilexnorm2026-test", private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/12 [00:00<?, ?ba/s]

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


Upload 0 LFS files: 0it [00:00, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/weerayut/multilexnorm2026-test/commit/9a6c04837da4abe7db1a7ff83f052dbdc183b2dd', commit_message='Upload dataset', commit_description='', oid='9a6c04837da4abe7db1a7ff83f052dbdc183b2dd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/weerayut/multilexnorm2026-test', endpoint='https://huggingface.co', repo_type='dataset', repo_id='weerayut/multilexnorm2026-test'), pr_revision=None, pr_num=None)

## Data

In [25]:
from datasets import load_dataset

pub_data = load_dataset("weerayut/multilexnorm2026-pub")
test_data = load_dataset("weerayut/multilexnorm2026-test")

pub_data, test_data

(DatasetDict({
     train: Dataset({
         features: ['raw', 'norm', 'lang'],
         num_rows: 39178
     })
     validation: Dataset({
         features: ['raw', 'norm', 'lang'],
         num_rows: 8408
     })
 }),
 DatasetDict({
     test: Dataset({
         features: ['raw', 'norm', 'lang'],
         num_rows: 11956
     })
 }))

In [ ]:
lang = "en"
en_train = pub_data["train"].filter(lambda x: x["lang"] == lang)
en_validation = pub_data["validation"].filter(lambda x: x["lang"] == lang)

pd.DataFrame(en_train).head(8)

,raw,norm,lang
0,"[rt, @teddyferrari1, :, "", ah, ..., @datzmenon...","[rt, @teddyferrari1, :, "", ah, ..., @datzmenon...",en
1,"[u, have, a, very, sexy, header, @jaibrooks1, ...","[you, have, a, very, sexy, header, @jaibrooks1...",en
2,"[i, miss, u, my, bie, !, where, u, wanna, out,...","[i, miss, you, my, bie, !, where, you, want to...",en
3,"["", cantik, ., rt, @historyinpics, :, julie, c...","["", cantik, ., rt, @historyinpics, :, julie, c...",en
4,"[rt, @fvckxhemmings, :, did, calum, slip, ?!!,...","[rt, @fvckxhemmings, :, did, calum, slip, ?!!,...",en
5,"[@daviddarkshade, and, you, didnt, make, this,...","[@daviddarkshade, and, you, didn't, make, this...",en
6,"[@xoxoxyareli, @billmillerprobz, pshh, i, went...","[@xoxoxyareli, @billmillerprobz, pshh, i, went...",en
7,"[@tindency, syempre, in, the, right, age, hahaha]","[@tindency, syempre, in, the, right, age, hahaha]",en
